# 과제 ③ 다중 벡터 검색기 + 최종 체인 — 연습본


색인 카드(요약문)를 **서랍**에 꽂고, 원본을 **서가**에 보관해 연결합니다. 오늘 수업의 마지막 조각입니다.

---

## 오늘 만드는 것

```
질문 ──▶ 요약으로 검색 ──▶ 원본 꺼내기 ──▶ 멀티모달 LLM ──▶ 답변
         (색인 카드)       (진짜 책)        (눈 달린 모델)
```

In [39]:
# ── 0. 결과 복원 ──────────────────────────────────────────
# 1) 필요한 모듈 import: os, re, time, uuid, pickle, pathlib.Path
# 2) python-dotenv 의 load_dotenv import
import os     #운영체제 기능(환경변수, 파일, 폴더 다루기 위한 기능)
import re     #정규표현식으로 문자열 패턴, 만든 검사, 검색,치환
import time   # 현재 시각 확인, 시간 측정, 대기 (체인 답변 생성 시간 측정)
import uuid   # 겹치지 않는 고유 ID 생성  uuid.uuid4() 요약본, 원본을 doc_id로 페어링
import pickle  #파이썬 객체를 파일로 저장, 복원
from pathlib import Path  #경로를 객체로 다룰 때 사용하는 기능
from dotenv import load_dotenv

# 3) .env 가 있는 폴더를 프로젝트 루트로 자동 탐색
#    - Path.cwd() 와 그 상위 폴더(parents)를 차례로 확인해 (p / ".env") 가 존재하는 첫 경로를 PROJECT_ROOT 로 지정
#    - 찾지 못하면 Path.cwd() 를 사용
#    - PROJECT_ROOT 출력 → c:\workspaces\multimodal-rag 이면 정상

PROJECT_ROOT =  next(
    ( p for p in [Path.cwd(), *Path.cwd().parents] if(p / ".env").exists()), Path.cwd(),
)
print("프로젝트 루트 : ", PROJECT_ROOT)

# 4) PROJECT_ROOT 의 .env 파일 로드 (load_dotenv)
load_dotenv(".env")

# 5) OPENAI_API_KEY 가 없으면 ValueError 발생시키기
try:
    openai_api_key = os.environ["OPENAI_API_KEY"]
    if not openai_api_key.strip():
        raise ValueError("OPENAI_API_KEY 값이 비어 있습니다. .env파일 확인해 주세요")
except KeyError:
     raise ValueError("OPENAI_API_KEY 키가 없습니다. .env파일을 확인해 주세요")

# 6) processed 경로 = PROJECT_ROOT / "data" / "processed"
processed = PROJECT_ROOT / "data" / "processed_assignment"
# 7) 추출 결과 복원: processed/"extracted.pkl" 을 pickle 로 읽기 ("rb")
#    - texts, tables, texts_2k_token 꺼내기
with open(processed/"extracted.pkl","rb") as f:
   doc1 =  pickle.load(f)
   texts, tables, texts_2k_token =  doc1["texts"] , doc1["tables"],doc1["texts_2k_token"]

# 8) 요약 결과 복원: processed/"summaries.pkl" 을 pickle 로 읽기
#    - text_summaries, table_summaries, image_summaries, img_base64_list 꺼내기
with open(processed/"summaries.pkl","rb") as f1:
    doc2 = pickle.load(f1)
    text_summaries = doc2["text_summaries"]
    table_summaries = doc2["table_summaries"]
    image_summaries = doc2["image_summaries"]
    img_base64_list = doc2["img_base64_list"]

# 9) "복원 완료" 와 함께 텍스트·표·이미지의 원본 수 / 요약 수 출력
print("복원 완료")
print(f" 텍스트 원본 {len(texts_2k_token):3d}건  / 요약{len(text_summaries):3d} 건")
print(f" 테이블 원본 {len(tables):3d}건  / 요약{len(table_summaries):3d} 건")
print(f" 이미지 원본 {len(img_base64_list):3d}건  / 요약{len(image_summaries):3d} 건")

프로젝트 루트 :  /Users/huyn/Project/multimodal-rag
복원 완료
 텍스트 원본   3건  / 요약  3 건
 테이블 원본  12건  / 요약 12 건
 이미지 원본   7건  / 요약  7 건


# 2. 검색기 만들기

In [40]:
# ── 2. 검색기 만들기 ──────────────────────────────────────
# langchain 1.x 에서 두 클래스의 위치가 바뀌었습니다.
# 어느 버전을 설치했든 동작하도록 두 경로를 모두 시도합니다. (try / except ImportError)

# 1) MultiVectorRetriever import
#    - langchain 1.x : langchain_classic.retrievers.multi_vector
#    - langchain 0.3 : langchain.retrievers.multi_vector
try:
    from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
except:
      print("MultiVectorRetriever 클래스가 필요합니다.")

# 2) InMemoryStore import
#    - langchain 1.x : langchain_core.stores
#    - langchain 0.3 : langchain.storage
try:
     from langchain_core.stores import InMemoryStore
except:
     print("InMemoryStore 클래스가 필요합니다.")

# 3) Chroma(langchain_chroma), OpenAIEmbeddings(langchain_openai) import
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

# 4) 요약문이 들어갈 벡터 DB (= 색인 카드 서랍)
#    - collection_name="multimodal_rag"
#    - embedding_function = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma(
     collection_name="multimodal_rag" , embedding_function = OpenAIEmbeddings(model="text-embedding-3-small")
)
vectorstore.reset_collection()


# 5) 원본이 들어갈 저장소 (= 서가) : InMemoryStore()
store = InMemoryStore()

# 6) 연결 키(= 청구기호) : id_key = "doc_id"
id_key = "doc_id"
# 7) MultiVectorRetriever(vectorstore=..., docstore=..., id_key=...) 생성
retriever = MultiVectorRetriever(
     vectorstore= vectorstore, docstore= store, id_key = id_key
)


# 8) "검색기 생성 완료" 출력
print("검색기 생성 완료")

검색기 생성 완료


# 3. 요약과 원본을 같은 ID로 짝짓기

In [41]:
# ── 3. 요약과 원본을 같은 ID로 짝짓기 ─────────────────────
# 1) langchain_core.documents 의 Document import
from langchain_core.documents  import Document as _Doc

# 2) add_documents(retriever, doc_summaries, doc_contents) 함수 정의
#    """요약본은 벡터 DB에, 원본은 docstore에 '같은 id'로 저장합니다."""
#    - 원본 개수만큼 uuid4 로 doc_ids 생성 (문자열로 변환)
#    - 요약문마다 Document(page_content=요약, metadata={id_key: doc_ids[i]}) 생성
def add_documents(retriever, doc_summaries, doc_contents):
    doc_ids = []
    # for _ in doc_contents:
    #     new_id = str(uuid.uuid4())
    #     doc_ids.append(new_id)
    doc_ids = [str(uuid.uuid4()) for _ in doc_contents]

    summary_docs = [
        _Doc(page_content= s, metadata={id_key:doc_ids[i]})
        for i ,s in enumerate(doc_summaries)
    ]

    #    - TODO 1: 요약본은 어디에 넣을까요? (vectorstore / docstore)
    #      → retriever.vectorstore.add_documents(...)
    retriever.vectorstore.add_documents(summary_docs)
    #    - TODO 2: 원본은 어디에 넣을까요?
    retriever.docstore.mset(list(zip(doc_ids, doc_contents)))

# 3) 데이터가 있을 때만 각각 저장
#    - text_summaries  ↔ texts_2k_token
#    - table_summaries ↔ tables
#    - image_summaries ↔ img_base64_list
if text_summaries:
    add_documents(retriever, text_summaries, texts_2k_token)
if table_summaries:
    add_documents(retriever, table_summaries, tables)
if image_summaries:
    add_documents(retriever, image_summaries, img_base64_list)

# 4) "색인 완료" 와 벡터 DB에 저장된 요약 수 출력 (vectorstore.get()["ids"] 의 길이)
# ⚠️ 이 셀을 두 번 실행하면 데이터가 중복됩니다.
print("색인 완료:", len(retriever.vectorstore.get()["ids"]), "건")

색인 완료: 22 건


# 4. 검색 테스트 - 요약으로 찾고 원본을 받는다

In [42]:
# ── 4. 검색 테스트 ────────────────────────────────────────
# TODO 3: 그래프나 표를 봐야만 답할 수 있는 질문을 query 에 적으세요
query = "2025년 9월 30일 기준 현금 및 현금성 자산과 총자산은 각각 얼마인가요?"


# 1) retriever.invoke(query) 결과를 found 에 저장
# 2) 검색된 원본 수 출력
found = retriever.invoke(query)
# 1. 질문을 임베딩으로 바꾼다.
# 2. vectorstore에서 질문과 가장 비슷한 요약문을 찾는다.(Chroma)
# 3. 찾은 요약의 metadata["doc_id"] 읽는다.
# 4. doc_id(번호표) docstore 에서 원본을 꺼내 반환한다. (Inmemory Store)

print("검색된 원본 수:", len(found))
print()

# 3) found 를 순회하며
#    - content = doc.page_content (Document 가 아니면 doc 그대로)
#    - 정규식 "^[A-Za-z0-9+/]{100,}={0,2}$" 에 맞으면 "이미지(Base64)", 아니면 "텍스트/표"
#    - 종류, 길이, 앞 120자(줄바꿈 제거) 출력

for i, doc in enumerate(found):
    content = doc.page_content if hasattr(doc, "page_content") else doc
    is_image = bool(re.match("^[A-Za-z0-9+/]{100,}={0,2}$", content))
    kind = "이미지(Base64)" if is_image else "텍스트/표"
    print(f"[{i}] {kind} — 길이 {len(content):,}자")
    print("   ", content[:120].replace("\n", " "), "...")
    print()

검색된 원본 수: 4

[0] 텍스트/표 — 길이 723자
    (in thousands) ASSETS Cash and cash equivalents Short-term investments Accounts receivable Deferred cost of revenues Pre ...

[1] 텍스트/표 — 길이 841자
    Nine Months Ended September 30, (in thousands) 2024 2025 Cash flows from operating activities: Net income (1) $74,667 $3 ...

[2] 텍스트/표 — 길이 365자
    Three Months Ended September 30, (in thousands) 2024 2025 Net cash provided by operating activities $56,267 $84,239 Less ...

[3] 텍스트/표 — 길이 339자
    (in thousands) Total GAAP Operating Expense Less: Depreciation and amortization Less: Stock-based compensation expenses  ...



# 5. 검색 결과를 텍스트와 이미지로 분리

In [43]:
# ── 5. 검색 결과를 텍스트와 이미지로 분리 ─────────────────
# 1) langchain_core.documents 의 Document import
from langchain_core.documents import Document

# 2) split_image_text_types(docs) 함수 정의
#    """검색 결과를 Base64 이미지와 텍스트로 나눕니다."""
#    - b64_images, texts_out 빈 리스트 준비
#    - Document 면 page_content 를 꺼내 문자열로 만들기
#    - Base64 문자열은 영문/숫자/+/= 만으로 길게 이어집니다
#      → 정규식 "^[A-Za-z0-9+/]{100,}={0,2}$" 매칭 시 이미지, 아니면 텍스트
#    - {"images": b64_images, "texts": texts_out} 반환

def split_image_text_types(docs):
    b64_images = []
    texts_out = []

    for doc in docs:
      content = doc.page_content if isinstance(doc, Document) else doc
      if re.match("^[A-Za-z0-9+/]{100,}={0,2}$", content):
            b64_images.append(content)
      else:
            texts_out.append(content)


    return {"images" : b64_images, "texts": texts_out}
# 3) 앞서 검색한 결과(found)로 동작 확인 → 이미지 / 텍스트 건수 출력

split_data = split_image_text_types(found)
print("검색 결과 분리 완료")
print(" 이미지(Base64) 수:", len(split_data["images"]))
print("이미지(Base64) 길이 합계:", sum(len(s) for s in split_data["images"]))
print(" 텍스트/표 수:", len(split_data["texts"]))

검색 결과 분리 완료
 이미지(Base64) 수: 0
이미지(Base64) 길이 합계: 0
 텍스트/표 수: 4


# 6. 최종 멀티모달 체인 만들기

In [44]:
# ── 6. 최종 멀티모달 체인 만들기 ──────────────────────────
# 1) import
#    - HumanMessage (langchain_core.messages)
#    - StrOutputParser (langchain_core.output_parsers)
#    - ChatPromptTemplate (langchain_core.prompts)
#    - RunnableLambda, RunnablePassthrough (langchain_core.runnables)
#    - ChatOpenAI (langchain_openai)
from langchain_core.messages import HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI


# 2) img_prompt_func(data_dict) 함수 정의
#    """검색된 원본을 멀티모달 LLM이 읽을 형태로 조립합니다."""
#    - messages 빈 리스트 준비
#    - 1) 이미지를 먼저 넣습니다 (모델이 그림을 먼저 보게)
#         data_dict["context"]["images"] 마다 {"type": "image_url", "image_url": {"url": ...}} 추가
#         TODO 4: Base64 앞에 붙일 접두사를 완성하세요 (data:image/jpeg;base64,)
#    - 2) 텍스트 컨텍스트와 질문을 넣습니다
#         texts 를 줄바꿈으로 합치고 {"type": "text", "text": ...} 추가
#         지시문: "아래 자료를 근거로 질문에 한국어로 답하세요. 자료에 없는 내용은 모른다고 답하세요."
#         + 질문(data_dict["question"]) + 참고 텍스트
#    - [HumanMessage(content=messages)] 반환
def img_prompt_func(data_dict):
    messages = []

    # 1 이미지를 먼저 넣는다.
    for image in data_dict["context"]["images"]:
        messages.append(
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{image}"},
            }
        )
    # 2 텍스트 컨텍스트와 질문을 넣는다.
    formatted_texts = "\n".join(data_dict["context"]["texts"])
    messages.append(
        {
            "type": "text",
            "text": (
                "자료를 근거로 질문에 한국어로 답하세요\n"
                "그래프로 나타난 수치는 이미지를 분석하여 대략적인 값을 요약에 포함하세요\n"
                "자료에 없는 내용은 모른다고 답하세요.\n\n"
                f"질문: {data_dict['question']}\n\n"
                f"참고 텍스트 :\n {formatted_texts}"
            ),
        }
    )
    return [HumanMessage(content=messages)]


# 3) 모델: ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024)
model = ChatOpenAI(temperature=0, model="gpt-4o-mini", max_tokens=1024)
# 4) 체인 조립 (LCEL)
#    {"context": retriever | RunnableLambda(split_image_text_types),
#     "question": RunnablePassthrough()}
#    | RunnableLambda(img_prompt_func)
#    | TODO 5: 멀티모달 LLM 변수
#    | StrOutputParser()
chain = (
    {
        "context": retriever | RunnableLambda(split_image_text_types),
        "question": RunnablePassthrough(),
    }
    | RunnableLambda(img_prompt_func)
    | model
    | StrOutputParser()
)

# 5) "최종 체인 조립 완료" 출력
print("mission complete!")

mission complete!


# 7. 질의 실행

In [45]:
# ── 7. 질의 실행 ──────────────────────────────────────────
# 1) 시작 시간 기록 (time.time())
start = time.time()

# 2) chain.invoke(query) 결과를 answer 에 저장
answer = chain.invoke(query)

# 3) 소요 시간, 질문(Q), 구분선, 답변 출력
print(f"소요 시간: {time.time() - start:.1f}초")
print("Q:", query)
print("-" * 60)
print(answer)

소요 시간: 1.6초
Q: 2025년 9월 30일 기준 현금 및 현금성 자산과 총자산은 각각 얼마인가요?
------------------------------------------------------------
2025년 9월 30일 기준으로 현금 및 현금성 자산은 약 1,011,542천 달러이며, 총자산은 약 1,885,638천 달러입니다.


# 8. Before / After 비교

In [46]:
# ── 8-1. 비교용 텍스트 전용 RAG ────────────────────────────
# 이미지와 표를 뺀 것 말고는 모든 조건이 같아야 합니다.

# 1) 텍스트만 들어간 벡터 DB 생성
#    - Chroma(collection_name="text_only_rag", embedding_function=OpenAIEmbeddings(model="text-embedding-3-small"))
#    - texts_2k_token 을 add_texts 로 저장
#    - as_retriever(search_kwargs={"k": 4}) 로 검색기 생성
text_only_vectorstore = Chroma(
    collection_name="text_only_rag", embedding_function=OpenAIEmbeddings(model="text-embedding-3-small")
)
text_only_vectorstore.add_texts(texts_2k_token)
text_only_retriever = text_only_vectorstore.as_retriever(search_kwargs={"k": 4})

# 2) 프롬프트: ChatPromptTemplate.from_template
#    - 멀티모달 체인과 같은 지시문 + {question} + {context}
text_only_prompt = ChatPromptTemplate.from_template(
    "자료를 근거로 질문에 한국어로 답하세요\n"
    "그래프로 나타난 수치는 이미지를 분석하여 대략적인 값을 요약에 포함하세요\n"
    "자료에 없는 내용은 모른다고 답하세요.\n\n"
    "질문: {question}\n\n"
    "참고 텍스트 :\n {context}"
)

# 3) 체인 조립
#    {"context": 검색기 | (문서들의 page_content 를 줄바꿈으로 합치는 함수),
#     "question": RunnablePassthrough()}
#    | 프롬프트 | model | StrOutputParser()
def join_docs(docs):
    return "\n".join(doc.page_content for doc in docs)

text_only_chain = (
    {
        "context": text_only_retriever | RunnableLambda(join_docs),
        "question": RunnablePassthrough(),
    }
    | text_only_prompt
    | model
    | StrOutputParser()
)

# 4) "비교용 텍스트 전용 RAG 준비 완료" 출력
print("비교용 텍스트 전용 RAG 준비 완료")

비교용 텍스트 전용 RAG 준비 완료


In [47]:
# ── 8-2. Before / After 출력 ───────────────────────────────
# 1) 구분선과 질문 출력
print("=" * 60)
print("Q:", query)
print("=" * 60)

# 2) [BEFORE] 텍스트 전용 RAG
#    - text_only_chain.invoke(query) 결과를 before 에 저장 후 출력
before = text_only_chain.invoke(query)
print("[BEFORE] 텍스트 전용 RAG")
print("-" * 60)
print(before)
print()

# 3) [AFTER] 멀티모달 RAG
#    - chain.invoke(query) 결과를 after 에 저장 후 출력
after = chain.invoke(query)
print("[AFTER] 멀티모달 RAG")
print("-" * 60)
print(after)
print()

# 4) "이 출력 전체를 캡처해 README.md에 넣으세요." 안내 출력
print("=" * 60)
print("이 출력 전체를 캡처해 README.md에 넣으세요.")

Q: 2025년 9월 30일 기준 현금 및 현금성 자산과 총자산은 각각 얼마인가요?
[BEFORE] 텍스트 전용 RAG
------------------------------------------------------------
자료에 따르면, 2025년 9월 30일 기준으로 현금 및 현금성 자산과 총자산에 대한 구체적인 수치는 제공되지 않았습니다. 따라서 해당 정보는 알 수 없습니다.

[AFTER] 멀티모달 RAG
------------------------------------------------------------
2025년 9월 30일 기준으로 현금 및 현금성 자산은 약 1,011,542천 달러이며, 총자산은 약 1,885,638천 달러입니다.

이 출력 전체를 캡처해 README.md에 넣으세요.
